# Автоматическое построение признаков с Featuretools

Синтетические данные интернет-магазина: покупатели, заказы, товары и позиции заказов. Признаки строятся для покупателей с помощью DFS (`max_depth=2`).

In [1]:
import warnings

import pandas as pd
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore", message=r"pkg_resources is deprecated as an API.*",
        category=UserWarning, module=r"woodwork",
    )
    import featuretools as ft
from woodwork.logical_types import Categorical, Datetime, Double, Integer, NaturalLanguage

print(f"pandas: {pd.__version__}; featuretools: {ft.__version__}")

pandas: 2.3.3; featuretools: 1.31.0


In [2]:
customers = pd.DataFrame({
    "client_id": [1, 2, 3, 4, 5, 6],
    "name": ["Анна", "Иван", "Мария", "Олег", "Елена", "Павел"],
    "join_date": pd.to_datetime([
        "2025-01-05", "2025-01-12", "2025-02-03",
        "2025-02-15", "2025-03-01", "2025-03-10",
    ]),
    "region": ["Сибирь", "Центр", "Сибирь", "Урал", "Центр", "Урал"],
})

products = pd.DataFrame({
    "product_id": [101, 102, 103, 104, 105],
    "category": ["Книги", "Книги", "Электроника", "Дом", "Дом"],
    "price": [500.0, 750.0, 2500.0, 1200.0, 350.0],
})

orders = pd.DataFrame({
    "order_id": list(range(1001, 1013)),
    "client_id": [1, 2, 1, 3, 4, 2, 5, 3, 6, 4, 5, 6],
    "order_date": pd.to_datetime([
        "2025-01-15", "2025-01-20", "2025-02-10", "2025-02-12",
        "2025-02-20", "2025-03-05", "2025-03-08", "2025-03-15",
        "2025-03-20", "2025-04-03", "2025-04-10", "2025-04-18",
    ]),
})

order_items = pd.DataFrame([
    (1, 1001, 101, 2), (2, 1001, 104, 1),
    (3, 1002, 102, 1), (4, 1002, 105, 3),
    (5, 1003, 103, 1), (6, 1003, 101, 1),
    (7, 1004, 104, 2), (8, 1004, 105, 1),
    (9, 1005, 102, 2), (10, 1005, 103, 1),
    (11, 1006, 101, 3), (12, 1006, 105, 2),
    (13, 1007, 103, 1), (14, 1007, 104, 1),
    (15, 1008, 101, 1), (16, 1008, 102, 2),
    (17, 1009, 105, 4), (18, 1009, 104, 1),
    (19, 1010, 103, 2), (20, 1010, 105, 1),
    (21, 1011, 102, 3), (22, 1011, 101, 2),
    (23, 1012, 104, 2), (24, 1012, 102, 1),
], columns=["item_id", "order_id", "product_id", "quantity"])

item_prices = order_items.merge(products[["product_id", "price"]], on="product_id")
order_totals = (
    item_prices.assign(amount=item_prices["price"] * item_prices["quantity"])
    .groupby("order_id")["amount"].sum()
)
orders["total_amount"] = orders["order_id"].map(order_totals)

for table_name, table in {
    "customers": customers, "orders": orders,
    "products": products, "order_items": order_items,
}.items():
    print(f"{table_name}: {table.shape}")
    display(table.head())

customers: (6, 4)


,client_id,name,join_date,region
0,1,Анна,2025-01-05,Сибирь
1,2,Иван,2025-01-12,Центр
2,3,Мария,2025-02-03,Сибирь
3,4,Олег,2025-02-15,Урал
4,5,Елена,2025-03-01,Центр


orders: (12, 4)


,order_id,client_id,order_date,total_amount
0,1001,1,2025-01-15,2200.0
1,1002,2,2025-01-20,1800.0
2,1003,1,2025-02-10,3000.0
3,1004,3,2025-02-12,2750.0
4,1005,4,2025-02-20,4000.0


products: (5, 3)


,product_id,category,price
0,101,Книги,500.0
1,102,Книги,750.0
2,103,Электроника,2500.0
3,104,Дом,1200.0
4,105,Дом,350.0


order_items: (24, 4)


,item_id,order_id,product_id,quantity
0,1,1001,101,2
1,2,1001,104,1
2,3,1002,102,1
3,4,1002,105,3
4,5,1003,103,1


`EntitySet` объединяет таблицы по ключам. Даты регистрации и заказа задаются как временные индексы, регион и категория товара — как категориальные признаки. Имя покупателя исключено из генерации.

In [3]:
es = ft.EntitySet(id="shop")
es = es.add_dataframe(
    dataframe_name="customers", dataframe=customers,
    index="client_id", time_index="join_date",
    logical_types={"client_id": Integer, "name": NaturalLanguage,
        "join_date": Datetime, "region": Categorical},
)
es = es.add_dataframe(
    dataframe_name="orders", dataframe=orders,
    index="order_id", time_index="order_date",
    logical_types={
        "order_id": Integer, "client_id": Integer,
        "order_date": Datetime, "total_amount": Double,
    },
)
es = es.add_dataframe(
    dataframe_name="products", dataframe=products, index="product_id",
    logical_types={"product_id": Integer, "category": Categorical, "price": Double},
)
es = es.add_dataframe(
    dataframe_name="order_items", dataframe=order_items, index="item_id",
    logical_types={
        "item_id": Integer, "order_id": Integer,
        "product_id": Integer, "quantity": Integer,
    },
)
es = es.add_relationship(
    parent_dataframe_name="customers", parent_column_name="client_id",
    child_dataframe_name="orders", child_column_name="client_id",
)
es = es.add_relationship(
    parent_dataframe_name="orders", parent_column_name="order_id",
    child_dataframe_name="order_items", child_column_name="order_id",
)
es = es.add_relationship(
    parent_dataframe_name="products", parent_column_name="product_id",
    child_dataframe_name="order_items", child_column_name="product_id",
)
es

Entityset: shop
  DataFrames:
    customers [Rows: 6, Columns: 4]
    orders [Rows: 12, Columns: 4]
    products [Rows: 5, Columns: 3]
    order_items [Rows: 24, Columns: 4]
  Relationships:
    orders.client_id -> customers.client_id
    order_items.order_id -> orders.order_id
    order_items.product_id -> products.product_id

In [4]:
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message=r"The provided callable .* is currently using SeriesGroupBy.*",
        category=FutureWarning,
        module=r"featuretools\.computational_backends\.feature_set_calculator",
    )
    feature_matrix, feature_defs = ft.dfs(
        entityset=es,
        target_dataframe_name="customers",
        max_depth=2,
        agg_primitives=["sum", "mean", "count", "max", "min", "num_unique"],
        trans_primitives=["month", "weekday"],
        ignore_columns={"customers": ["name"]},
    )

print(f"Размер матрицы признаков: {feature_matrix.shape}")
print("Список признаков:")
for feature in feature_defs:
    print(feature.get_name())

Размер матрицы признаков: (6, 35)
Список признаков:
region
COUNT(orders)
MAX(orders.total_amount)
MEAN(orders.total_amount)
MIN(orders.total_amount)
SUM(orders.total_amount)
COUNT(order_items)
MAX(order_items.quantity)
MEAN(order_items.quantity)
MIN(order_items.quantity)
SUM(order_items.quantity)
MONTH(join_date)
WEEKDAY(join_date)
MAX(orders.COUNT(order_items))
MAX(orders.MEAN(order_items.quantity))
MAX(orders.MIN(order_items.quantity))
MAX(orders.SUM(order_items.quantity))
MEAN(orders.COUNT(order_items))
MEAN(orders.MAX(order_items.quantity))
MEAN(orders.MEAN(order_items.quantity))
MEAN(orders.MIN(order_items.quantity))
MEAN(orders.SUM(order_items.quantity))
MIN(orders.COUNT(order_items))
MIN(orders.MAX(order_items.quantity))
MIN(orders.MEAN(order_items.quantity))
MIN(orders.SUM(order_items.quantity))
NUM_UNIQUE(orders.MONTH(order_date))
NUM_UNIQUE(orders.WEEKDAY(order_date))
SUM(orders.MAX(order_items.quantity))
SUM(orders.MEAN(order_items.quantity))
SUM(orders.MIN(order_items.quant

In [5]:
display(feature_matrix.head())

print("Количество признаков по глубине:")
for depth in sorted({feature.get_depth() for feature in feature_defs}):
    count = sum(feature.get_depth() == depth for feature in feature_defs)
    print(f"Глубина {depth}: {count}")

,region,COUNT(orders),MAX(orders.total_amount),MEAN(orders.total_amount),MIN(orders.total_amount),SUM(orders.total_amount),COUNT(order_items),MAX(order_items.quantity),MEAN(order_items.quantity),MIN(order_items.quantity),...,MIN(orders.SUM(order_items.quantity)),NUM_UNIQUE(orders.MONTH(order_date)),NUM_UNIQUE(orders.WEEKDAY(order_date)),SUM(orders.MAX(order_items.quantity)),SUM(orders.MEAN(order_items.quantity)),SUM(orders.MIN(order_items.quantity)),MAX(order_items.orders.total_amount),MEAN(order_items.orders.total_amount),MIN(order_items.orders.total_amount),SUM(order_items.orders.total_amount)
client_id,,,,,,,,,,,,,,,,,,,,,
1,Сибирь,2,3000.0,2600.0,2200.0,5200.0,4,2.0,1.25,1.0,...,2.0,2,2,3.0,2.5,2.0,3000.0,2600.0,2200.0,10400.0
2,Центр,2,2200.0,2000.0,1800.0,4000.0,4,3.0,2.25,1.0,...,4.0,2,2,6.0,4.5,3.0,2200.0,2000.0,1800.0,8000.0
3,Сибирь,2,2750.0,2375.0,2000.0,4750.0,4,2.0,1.50,1.0,...,3.0,2,2,4.0,3.0,2.0,2750.0,2375.0,2000.0,9500.0
4,Урал,2,5350.0,4675.0,4000.0,9350.0,4,2.0,1.50,1.0,...,3.0,2,1,4.0,3.0,2.0,5350.0,4675.0,4000.0,18700.0
5,Центр,2,3700.0,3475.0,3250.0,6950.0,4,3.0,1.75,1.0,...,2.0,2,2,4.0,3.5,3.0,3700.0,3475.0,3250.0,13900.0


Количество признаков по глубине:
Глубина 0: 1
Глубина 1: 12
Глубина 2: 22


Для каждого покупателя получена строка признаков: характеристики регистрации, статистики заказов и вложенные агрегаты по позициям. Глубина 2 позволяет учитывать цепочку «покупатель → заказ → позиции заказа». Все таблицы и связи включены в `EntitySet`; более длинные цепочки ограничены заданной глубиной DFS.